In [ ]:
import sys
sys.path.insert(0, '../../src')

import numpy as np
import pandas as pd
import statsmodels.formula.api as smf
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

np.random.seed(42)
from mmm_lab.data_generation.baseline import generate_baseline_geo_data
from mmm_lab.data_generation.marketing import add_marketing_effects, geometric_adstock, hill_saturation

baseline_df = generate_baseline_geo_data(n_geos=40, n_weeks=104, start_date='2023-01-01')
df = add_marketing_effects(baseline_df, channels=['tv', 'paid_search'])
df['month'] = df['date'].dt.month

truth_tv = df['effect_tv'].sum() / df['spend_tv'].sum()
truth_ps = df['effect_paid_search'].sum() / df['spend_paid_search'].sum()
print(f"Ground truth ROAS — TV: {truth_tv:.3f}, PS: {truth_ps:.3f}")


In [ ]:
def compute_adstock_per_geo(df, col, decay, K, S):
    """Apply geometric adstock + Hill saturation per geo. Returns series aligned to df index."""
    result = pd.Series(index=df.index, dtype=float)
    for geo_id, gdf in df.groupby('geo'):
        ads = geometric_adstock(gdf[col].values, decay)
        sat = hill_saturation(ads, K, S)
        result.loc[gdf.index] = sat
    return result

results = {}

def record(name, tv, ps, r2, notes=''):
    results[name] = {
        'tv': tv, 'ps': ps,
        'tv_err': (tv - truth_tv) / truth_tv,
        'ps_err': (ps - truth_ps) / truth_ps,
        'r2': r2
    }
    print(f"{name:<50}  TV: {tv:.3f} ({(tv-truth_tv)/truth_tv:+.1%})  "
          f"PS: {ps:.3f} ({(ps-truth_ps)/truth_ps:+.1%})  R²: {r2:.3f}"
          + (f"  [{notes}]" if notes else ""))

print(f"{'Model':<50}  {'TV ROAS':>18}  {'PS ROAS':>18}  {'R²':>6}")
print("-" * 100)


In [ ]:
m = smf.ols('total_bookings ~ spend_tv + spend_paid_search', data=df).fit()
record('1. Naive OLS (no controls, no FE)',
       m.params['spend_tv'], m.params['spend_paid_search'], m.rsquared,
       notes='cross-sectional confound: big geos spend more AND buy more')


In [ ]:
m = smf.ols('total_bookings ~ spend_tv + spend_paid_search + C(geo)', data=df).fit()
record('2. Geo FE',
       m.params['spend_tv'], m.params['spend_paid_search'], m.rsquared,
       notes='removes cross-geo confound; identifies from within-geo temporal variation')


In [ ]:
m = smf.ols('total_bookings ~ spend_tv + spend_paid_search + demand_very_good + C(geo)', data=df).fit()
record('3. Geo FE + demand control (r=0.97)',
       m.params['spend_tv'], m.params['spend_paid_search'], m.rsquared)


In [ ]:
m = smf.ols('total_bookings ~ spend_tv + spend_paid_search + demand_very_good + C(geo) + C(month)', data=df).fit()
record('4. Geo FE + month FE + demand control',
       m.params['spend_tv'], m.params['spend_paid_search'], m.rsquared,
       notes='seasonal dummies remove cyclical confounds without absorbing all temporal media variation')


In [ ]:
df['week_id'] = (df['date'].dt.year - df['date'].dt.year.min()) * 53 + df['date'].dt.isocalendar().week.astype(int)
m = smf.ols('total_bookings ~ spend_tv + spend_paid_search + C(geo) + C(week_id)', data=df).fit()
record('5. Geo FE + week FE (identification collapse)',
       m.params['spend_tv'], m.params['spend_paid_search'], m.rsquared,
       notes='week dummies absorb all national temporal variation; identifies from geo cross-section only')


In [ ]:
df_lag = df.copy()
for lag in range(1, 5):
    df_lag[f'tv_lag{lag}']  = df_lag.groupby('geo')['spend_tv'].shift(lag)
    df_lag[f'ps_lag{lag}']  = df_lag.groupby('geo')['spend_paid_search'].shift(lag)
df_lag = df_lag.dropna(subset=[f'tv_lag{i}' for i in range(1,5)])

lag_terms = ' + '.join([f'tv_lag{i} + ps_lag{i}' for i in range(1, 5)])
formula = f'total_bookings ~ spend_tv + spend_paid_search + {lag_terms} + demand_very_good + C(geo)'
m = smf.ols(formula, data=df_lag).fit()

tv_cols = ['spend_tv']  + [f'tv_lag{i}' for i in range(1, 5)]
ps_cols = ['spend_paid_search'] + [f'ps_lag{i}' for i in range(1, 5)]
tv_roas = sum(m.params.get(c, 0) for c in tv_cols)
ps_roas = sum(m.params.get(c, 0) for c in ps_cols)
record('6. Distributed lags 0-4 + Geo FE + demand control', tv_roas, ps_roas, m.rsquared,
       notes='sum of lag coefs = total ROAS across carryover window')

# Show individual lag coefficients
print("\n  Lag coefficients:")
for i, (tc, pc) in enumerate(zip(tv_cols, ps_cols)):
    print(f"    lag{i}: TV={m.params.get(tc,0):+.4f} (p={m.pvalues.get(tc,1):.3f})  "
          f"PS={m.params.get(pc,0):+.4f} (p={m.pvalues.get(pc,1):.3f})")


In [ ]:
# Keep only lag terms with p < 0.05; always include lag0
sig_tv = ['spend_tv']  + [f'tv_lag{i}' for i in range(1,5) if m.pvalues.get(f'tv_lag{i}', 1) < 0.05]
sig_ps = ['spend_paid_search'] + [f'ps_lag{i}' for i in range(1,5) if m.pvalues.get(f'ps_lag{i}', 1) < 0.05]
print(f"  Significant TV lags: {sig_tv}")
print(f"  Significant PS lags: {sig_ps}")

sig_terms = ' + '.join(sig_tv[1:] + sig_ps[1:])  # lag0 already in base formula
formula_sig = f'total_bookings ~ spend_tv + spend_paid_search'
if sig_terms:
    formula_sig += f' + {sig_terms}'
formula_sig += ' + demand_very_good + C(geo)'

m_sig = smf.ols(formula_sig, data=df_lag).fit()
tv_roas = sum(m_sig.params.get(c, 0) for c in sig_tv)
ps_roas = sum(m_sig.params.get(c, 0) for c in sig_ps)
record('7. Stat-sig lags only + Geo FE + demand control', tv_roas, ps_roas, m_sig.rsquared)


In [ ]:
def counterfactual_roas(model, df, tv_feat, ps_feat, spend_tv='spend_tv', spend_ps='spend_paid_search'):
    """ROAS = (predicted with spend - predicted with spend=0) / total spend"""
    pred_actual = model.predict(df)
    
    df_no_tv = df.copy()
    df_no_tv[tv_feat] = 0
    tv_incremental = (pred_actual - model.predict(df_no_tv)).sum()
    
    df_no_ps = df.copy()
    df_no_ps[ps_feat] = 0
    ps_incremental = (pred_actual - model.predict(df_no_ps)).sum()
    
    return tv_incremental / df[spend_tv].sum(), ps_incremental / df[spend_ps].sum()


In [ ]:
# log1p of adstock — oracle decay rates, no Hill (log handles diminishing returns)
df['log_ads_tv'] = np.log1p(compute_adstock_per_geo(df, 'spend_tv',          decay=0.5, K=1, S=1))
df['log_ads_ps'] = np.log1p(compute_adstock_per_geo(df, 'spend_paid_search',  decay=0.3, K=1, S=1))

m = smf.ols('total_bookings ~ log_ads_tv + log_ads_ps + demand_very_good + C(geo)', data=df).fit()
tv, ps = counterfactual_roas(m, df, 'log_ads_tv', 'log_ads_ps')
record('11. log(adstock) oracle decay + Geo FE + control', tv, ps, m.rsquared)
